# Phase 3: Data Preparation (Cleaning)
In this notebook, we address data quality issues discovered during EDA, specifically handling duplicates, imputing missing values, and treating outliers using the Interquartile Range (IQR) method.

In [1]:
import pandas as pd
import numpy as np
import joblib

# Load the raw data again
df = pd.read_csv('../data/personal-loan.csv')

In [2]:
# Check for and drop duplicates
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dropped {initial_rows - df.shape[0]} duplicate rows.")

Dropped 0 duplicate rows.


In [3]:
# Columns identified with missing values during EDA
missing_cols = ['age', 'yrs_experience', 'family_size', 'income']

# Impute missing values with the median
for col in missing_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Verify imputation
print("Missing values after imputation:")
print(df[missing_cols].isnull().sum())

Missing values after imputation:
age               0
yrs_experience    0
family_size       0
income            0
dtype: int64


In [4]:
# Select continuous numerical columns to check for outliers
continuous_cols = ['age', 'yrs_experience', 'income', 'mortgage_amt', 'credit_card_spend']

# Implement the IQR outlier replacement loop
for col in continuous_cols:
    median_val = df[col].quantile(0.50)
    percentile_25 = df[col].quantile(0.25)
    percentile_75 = df[col].quantile(0.75)
    iqr = percentile_75 - percentile_25
    
    lower = percentile_25 - (1.5 * iqr)
    upper = percentile_75 + (1.5 * iqr)
    
    # Replace anything outside bounds with the median using numpy
    df[col] = np.where((df[col] < lower) | (df[col] > upper), median_val, df[col])

print("Outliers successfully replaced with column medians.")

Outliers successfully replaced with column medians.


In [5]:
# Save the cleaned dataframe as a pickled object
joblib.dump(df, '../data/cleaned_data.pkl')
print("Cleaned data successfully pickled as 'cleaned_data.pkl'.")

Cleaned data successfully pickled as 'cleaned_data.pkl'.
